# Transformer reference — compare after you try

Keep the dashboard's **Inside the model** lab open beside this notebook. Watch attention and the feed-forward layer, then write their calculations here.

You will build a small, CPU-friendly character-level language model. It is randomly initialized: no pretrained weights or API are involved. Short teaching stories make it runnable; expect imperfect generated text.

**How to learn:** predict a shape or output → replace one TODO → run the cell and its check → explain why it worked. A failed assertion is a clue, not a reason to paste the whole answer. `Transformer_Reference.ipynb` contains the matching completed implementation.

**Where to run:** JupyterLab on your computer, a VS Code Python notebook, or Google Colab. A GPU is unnecessary for these defaults. AI autocomplete is optional; turn it off while working through a derivation if it gives away the answer.

Local setup (in the workshop folder):
```sh
python -m pip install -r requirements-notebook.txt
python -m jupyterlab Write_Your_Transformer.ipynb
```

In Colab, upload this notebook. The included text is embedded, so no extra file is required. To use your own permitted text later, upload it to the session and change `TEXT_PATH`. Keep an independent final test set before making claims about quality.

In [ ]:
# Run only if imports in the next cell fail (for example, in a new Colab runtime).
# In a notebook, uncomment this line and run it, then restart the kernel if asked.
# %pip install torch matplotlib

In [ ]:
import math
import json
import hashlib
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

torch.set_num_threads(1)
SEED = 7
CONFIG = dict(width=32, heads=4, layers=2, context=32)
BATCH_SIZE = 8
STEPS = 160
print('PyTorch:', torch.__version__)
print('CPU training; configuration:', CONFIG)

## 1 · Turn text into IDs


A character-level tokenizer assigns an integer to every character. The network receives IDs, not raw strings. Reserve the final 10% of characters **before sampling training windows**. Build the vocabulary from the training portion; ID 0 represents characters not seen there.

Write `encode_text`. It should return a 1-D `torch.long` tensor. Then print the first 20 IDs and decode them. The supplied corpus is tiny and contains related stories: a separate region is useful for this exercise but is not a robust independent benchmark. For real evaluation, split by source/document and inspect duplicates.

In [ ]:
TEXT_PATH = None  # Or a local UTF-8 file such as 'my-stories.txt'
BUILT_IN_TEXT = 'Maya opened the little blue notebook beside the window. She wanted to remember three things: buy bread, water the plant, and call her brother. Outside, the morning rain tapped gently against the glass. She wrote each task on a separate line and closed the book.\n\nAt the station, Arun found a red scarf on an empty seat. He took it to the ticket desk. The woman behind the desk put the scarf in a drawer and wrote down the time. Later that morning, a traveler returned and asked whether anyone had found a scarf. The woman smiled and opened the drawer.\n\nA small library stood at the end of the street. Every Friday, a child named Lina helped return books to their shelves. She sorted them by the labels on their covers. One book had lost its label. Instead of guessing, Lina asked the librarian where it belonged. They looked it up together.\n\nThe baker arrived before sunrise. He measured flour into a bowl, added water, and mixed the dough. The first loaf was too flat. He checked his notes and noticed that he had used too little yeast. On the next morning, he changed one ingredient and kept the others the same.\n\nNoah built a small wooden boat with his grandfather. They tested it in a bucket of water. It leaned to the left because one side was heavier. They moved a little weight to the right and tried again. This time the boat stayed upright. Noah drew a picture of the change in his notebook.\n\nA teacher placed three stones on a table. One was smooth, one was rough, and one was striped. She asked the class to describe them without naming their colors. The children talked about shape, size, and texture. They discovered that the same object could be described in several useful ways.\n\nThe garden gate would not close. A branch had fallen across the path during the night. Eva moved the branch and tried the gate again. It closed easily. She had first thought the lock was broken, but checking the path showed her the real problem.\n\nA bus driver checked the route before leaving the depot. A bridge was closed for repairs, so he followed the marked detour. At the next stop, he explained the delay to the waiting passengers. One passenger asked when the bridge would reopen. The driver said he did not know and pointed to the notice from the city.\n\nTwo friends made soup for dinner. They tasted it before adding salt. The first spoonful was mild, so they added a small pinch and tasted again. They knew that adding more was easy, but removing it would be difficult. The soup was ready when both agreed that the flavor was right.\n\nIn the workshop, a clock stopped at noon. The repairer checked the battery before opening the mechanism. The battery was empty. After replacing it, she watched the hands move for several minutes. She recorded the repair and returned the clock to its owner.\n\nA student wanted to grow beans near the kitchen window. He planted seeds in two pots and gave each the same amount of water. One pot stayed in the light and the other in a dark cupboard. Every morning he measured the stems. He wrote down what he saw instead of what he expected to see.\n\nSara organized the photographs from a family trip. She made a folder for each day and kept a copy of the original files. When she accidentally renamed the wrong folder, she could recover it from the copy. The extra preparation took a few minutes and saved an afternoon of work.\n\nA shopkeeper counted the apples before opening the shop. He wrote the count on a card. At closing time, he compared the remaining apples with the sales record. The numbers did not match because one damaged apple had been thrown away. He added a separate note so the record explained what happened.\n\nThe music teacher played a short tune on the piano. The student repeated it slowly, one phrase at a time. When she missed a note, they returned to that phrase. Playing the whole song faster did not fix the mistake. Careful practice of the difficult part helped more.\n\nOmar carried a map on his walk through the hills. At a fork in the path, he compared the signs with the map. The old wooden sign pointed toward a closed trail. A newer notice described a safe route around it. He followed the newer information and marked the change on his map.\n\nAt the animal shelter, each bowl had a name beside it. The volunteer checked the names before giving out food. Two dogs looked very similar, but their meals were different. Matching the right bowl to the right animal mattered more than finishing quickly.\n\nA team made a paper tower in the classroom. Their first design was tall but fell when the table moved. They widened the base and used less paper at the top. The second tower was shorter and more stable. They explained both the benefit and the tradeoff to the class.\n\nThe farmer watched clouds gather over the field. He put the tools inside the shed before the rain arrived. The weather report had suggested rain, but it had not promised an exact time. He treated the report as useful guidance and still looked at the sky.\n\nA child learned to ride a bicycle on a quiet path. At first, her father held the seat. Then he let go for a few seconds at a time. She practiced starting and stopping before riding farther. By the end of the afternoon, she could explain what helped her keep her balance.\n\nIn a small cafe, a customer asked for tea without sugar. The server repeated the request before preparing the drink. A printed order card helped the kitchen keep track of several cups at once. Before handing over the tea, the server checked the card one more time.\n\nThe museum guide showed visitors a clay pot with a missing handle. A drawing beside it showed one possible reconstruction. The guide explained which parts were original and which parts were guesses. The visitors could appreciate the object without confusing evidence with imagination.\n\nA runner kept a journal of her training. Some days she ran quickly, and other days she rested. One fast run did not show her whole progress. She compared several weeks and noticed that she could run the same distance with less effort.\n\nAfter a storm, neighbors worked together to clear the street. They first checked for fallen wires and called the utility company about one dangerous branch. Then they moved the smaller debris. They chose tasks that they could do safely and waited for help with the rest.\n\nThe final story begins in a quiet bookshop. A customer is looking for a story about a lighthouse but cannot remember the title. The bookseller asks about the cover and the characters. Together they narrow the search to three books. They open each one and find the familiar first sentence. The customer thanks the bookseller and takes the book home.\n\nThat evening, a family sits at a table with an unfinished puzzle. They sort the pieces by their edges and colors. A piece that seems to fit the sky actually belongs to the water. They turn it around, examine the pattern, and move it to another place. Slowly, the picture becomes clearer.\n'
text = Path(TEXT_PATH).read_text(encoding='utf-8') if TEXT_PATH else BUILT_IN_TEXT
split = int(len(text) * 0.9)
vocab = ['<UNK>'] + sorted(set(text[:split]))
lookup = {char: i for i, char in enumerate(vocab)}
print('Vocabulary size:', len(vocab))

In [ ]:
def encode_text(text, lookup):
    return torch.tensor([lookup.get(char, 0) for char in text], dtype=torch.long)

In [ ]:
data = encode_text(text, lookup)
train_data, val_data = data[:split], data[split:]
assert data.ndim == 1 and data.dtype == torch.long
assert encode_text('a?', {'a': 1}).tolist() == [1, 0]
assert min(len(train_data), len(val_data)) > CONFIG['context']
print('Training characters:', len(train_data), '| validation:', len(val_data))
print('First 20 IDs:', data[:20].tolist())
print('Decoded:', ''.join(vocab[i] for i in data[:20].tolist()))

## 2 · Build x and the shifted target y


For the sequence `c a t`, input `c a` has target `a t`. Every position predicts the **next** character. A batch has shape `[batch, context]`; targets have the same shape.

Write `make_batch`. Sample valid starting offsets. Return an input window and the window shifted one character to the right. Use the optional generator for repeatable evaluation without consuming training randomness.

In [ ]:
def make_batch(data, context, batch_size, generator=None):
    starts = torch.randint(len(data) - context, (batch_size,), generator=generator)
    x = torch.stack([data[i:i+context] for i in starts])
    y = torch.stack([data[i+1:i+context+1] for i in starts])
    return x, y

In [ ]:
x, y = make_batch(torch.arange(100), 8, 3)
assert x.shape == y.shape == (3, 8)
assert torch.equal(y, x + 1), 'Every target should be shifted one position'
print('Input row:', x[0].tolist())
print('Target row:', y[0].tolist())

## 3 · Write causal multi-head attention


Open **Inside the model → Attention** in the dashboard.

`qkv(x)` returns `[batch, time, 3 × width]`. Split the last dimension into query, key, and value tensors. Reshape each to `[batch, heads, time, head_width]`. Compute `q @ k.transpose(-2,-1) / sqrt(head_width)`.

Set scores above the diagonal to negative infinity **before** softmax. Softmax runs over keys (the last axis). Multiply those weights by values. Rejoin the heads and apply the output projection. `.contiguous()` makes the transposed layout compatible with `.view()`.

**Prediction:** changing the last input must leave all earlier outputs unchanged. This checks causal isolation, not language quality.

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, width, heads):
        super().__init__()
        assert width % heads == 0
        self.heads, self.head_width = heads, width // heads
        self.qkv = nn.Linear(width, 3 * width)
        self.output = nn.Linear(width, width)

    def forward(self, x):
        batch, time, width = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        def split_heads(t):
            return t.view(batch, time, self.heads, self.head_width).transpose(1, 2)
        q, k, v = map(split_heads, (q, k, v))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_width)
        future = torch.ones(time, time, device=x.device, dtype=torch.bool).triu(1)
        weights = scores.masked_fill(future, float('-inf')).softmax(dim=-1)
        mixed = (weights @ v).transpose(1, 2).contiguous()
        return self.output(mixed.view(batch, time, width))

In [ ]:
attention = CausalAttention(8, 2).eval()
x = torch.randn(2, 5, 8)
altered = x.clone(); altered[:, -1] += 10
assert attention(x).shape == x.shape
torch.testing.assert_close(attention(x)[:, :-1], attention(altered)[:, :-1])
print('Attention shape and causal isolation passed.')

## 4 · Write the feed-forward layer


Open **Inside the model → Feed-forward**. The dashboard uses 4 → 8 → 4 for readability; here use width → 4×width → width.

Write an expansion `nn.Linear`, GELU, and contraction `nn.Linear`. Each token uses the **same learned matrices independently**. Attention mixes token positions; this layer transforms features at one position. A nonlinear activation is essential: two affine layers alone are equivalent to one affine layer.

`nn.Linear(in_features,out_features)` computes `x @ weight.T + bias` for row-oriented batches. It operates on the last axis. Do not flatten time into the feature dimension.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.up = nn.Linear(width, 4 * width)
        self.down = nn.Linear(4 * width, width)

    def forward(self, x):
        return self.down(F.gelu(self.up(x)))

In [ ]:
ffn = FeedForward(8).eval()
x = torch.randn(2, 5, 8)
altered = x.clone(); altered[:, 2] += 3
assert ffn(x).shape == x.shape
torch.testing.assert_close(ffn(x)[:, :2], ffn(altered)[:, :2])
torch.testing.assert_close(ffn(x)[:, 3:], ffn(altered)[:, 3:])
assert not torch.allclose(ffn(x)[:, 2], ffn(altered)[:, 2])
print('FFN changes one token independently; shared weights preserve shape.')

## 5 · Connect the Transformer


A pre-norm block does two residual updates:

`x = x + attention(norm1(x))`

`x = x + ffn(norm2(x))`

The bypass carries the original residual vector; **do not add the normalized vector instead**. Add token and position embeddings before the blocks, then final normalization and vocabulary projection after them. Output shape is `[batch,time,vocabulary]`, so every position produces next-character scores.

Complete the two forward methods. Keep shapes visible by printing them while you debug.

In [ ]:
class Block(nn.Module):
    def __init__(self, width, heads):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(width), nn.LayerNorm(width)
        self.attention, self.ffn = CausalAttention(width, heads), FeedForward(width)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        return x + self.ffn(self.norm2(x))

class TinyTransformer(nn.Module):
    def __init__(self, vocabulary, width, heads, layers, context):
        super().__init__()
        self.context = context
        self.tokens = nn.Embedding(vocabulary, width)
        self.positions = nn.Embedding(context, width)
        self.blocks = nn.Sequential(*[Block(width, heads) for _ in range(layers)])
        self.norm = nn.LayerNorm(width)
        self.to_logits = nn.Linear(width, vocabulary)

    def forward(self, ids):
        position_ids = torch.arange(ids.shape[1], device=ids.device)
        x = self.tokens(ids) + self.positions(position_ids)
        return self.to_logits(self.norm(self.blocks(x)))

In [ ]:
torch.manual_seed(SEED)
model = TinyTransformer(len(vocab), **CONFIG).eval()
x, y = make_batch(train_data, CONFIG['context'], BATCH_SIZE)
logits = model(x)
assert logits.shape == (BATCH_SIZE, CONFIG['context'], len(vocab))
altered = x.clone(); altered[:, -1] = (altered[:, -1] + 1) % len(vocab)
torch.testing.assert_close(model(x)[:, :-1], model(altered)[:, :-1])
print('Logits:', list(logits.shape))
print('Trainable parameters:', sum(p.numel() for p in model.parameters()))

## 6 · Write one training update


The next-token objective is cross-entropy between logits and the shifted target. Flatten batch and time so cross-entropy sees a list of predictions.

Order matters: clear stale gradients → forward pass → loss → backward → clip → optimizer step. `backward()` computes derivatives; `step()` changes parameters. Return a plain number for logging, using `loss.item()`.

The check below verifies that weights really change, so printing a loss without an optimizer update will not pass.

In [ ]:
def update(model, optimizer, x, y):
    optimizer.zero_grad(set_to_none=True)
    logits = model(x)
    loss = F.cross_entropy(logits.flatten(0, 1), y.flatten())
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    return loss.item()

In [ ]:
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.003)
before = model.to_logits.weight.detach().clone()
loss = update(model, optimizer, x, y)
assert math.isfinite(loss)
assert not torch.equal(before, model.to_logits.weight)
print('One update changed weights. Loss:', round(loss, 4))

## 7 · Compare two real training runs

Before running, predict which learning rate will reduce validation loss faster. The helper resets the random seed before model initialization, so both runs start with identical weights and see the same sampled training windows. Validation uses its own generator. Inspect the measured curves and explain the outcome. A validation region is not a final test set; repeated tuning can overfit it too.

Change only the learning rates first. Later, change width, context length, or text in the setup cells and rerun all dependent cells from the top. Parameter count and task difficulty may then change, so make comparisons carefully.

In [ ]:
@torch.no_grad()
def evaluate(model):
    model.eval()
    generator = torch.Generator().manual_seed(1234)
    losses = []
    for _ in range(6):
        x, y = make_batch(val_data, CONFIG['context'], BATCH_SIZE, generator)
        losses.append(F.cross_entropy(model(x).flatten(0, 1), y.flatten()).item())
    model.train()
    return sum(losses) / len(losses)

def experiment(learning_rate, steps=STEPS):
    # Fresh model, identical initialization and training batch sequence for each run.
    torch.manual_seed(SEED)
    model = TinyTransformer(len(vocab), **CONFIG)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    history = [dict(step=0, train_loss=None, validation_loss=evaluate(model))]
    for step in range(1, steps + 1):
        x, y = make_batch(train_data, CONFIG['context'], BATCH_SIZE)
        loss = update(model, optimizer, x, y)
        if step % 20 == 0 or step == steps:
            row = dict(step=step, train_loss=loss, validation_loss=evaluate(model))
            history.append(row)
            print(f'lr={learning_rate}: step {step}, train={loss:.3f}, val={row["validation_loss"]:.3f}')
    return model, history

model_a, history_a = experiment(0.001)
model_b, history_b = experiment(0.003)
for label, history in [('lr 0.001', history_a), ('lr 0.003', history_b)]:
    plt.plot([r['step'] for r in history], [r['validation_loss'] for r in history], marker='o', label=label)
plt.xlabel('Training step'); plt.ylabel('Held-out cross-entropy')
plt.title('Same initialization and data, different learning rates')
plt.legend(); plt.grid(alpha=0.2); plt.show()
print('Inspect the actual curves. Do not assume a higher learning rate always wins.')

## 8 · Generate text one character at a time


Use the final position's logits, not every position. Divide by a positive temperature, apply softmax, sample one ID, append it, and repeat. Crop the context to the model's maximum length.

Temperature changes the sampling distribution; it does not retrain the model. Compare two temperatures with the same model and random seed. Expect broken words from this tiny run. Fluent fragments are not proof of broad language ability.

In [ ]:
@torch.no_grad()
def generate(model, temperature=0.8, length=200):
    if temperature <= 0:
        raise ValueError('Use a positive temperature')
    model.eval()
    torch.manual_seed(42)
    ids = train_data[:1].reshape(1, 1)
    for _ in range(length):
        logits = model(ids[:, -model.context:])[:, -1] / temperature
        next_id = torch.multinomial(logits.softmax(-1), num_samples=1)
        ids = torch.cat([ids, next_id], dim=1)
    return ''.join(vocab[i] for i in ids[0].tolist())

In [ ]:
print('Temperature 0.5:\n', generate(model_b, temperature=0.5))
print('\nTemperature 1.0:\n', generate(model_b, temperature=1.0))
assert generate(model_b, length=0) == vocab[train_data[0].item()]

## 9 · Keep your results

Save and reload your model weights, export real losses, and inspect them alongside the dashboard visualizations. This saved notebook artifact is for inference; the separate `train.py` saves optimizer and random state for resumable training.

In [ ]:
out = Path('notebook-run'); out.mkdir(exist_ok=True)
report = dict(config=CONFIG, history=history_b,
              text_sha256=hashlib.sha256(text.encode()).hexdigest(),
              note='Notebook run B; contiguous held-out teaching text, not a broad benchmark.')
(out / 'metrics.json').write_text(json.dumps(report, indent=2))
(out / 'sample.txt').write_text(generate(model_b), encoding='utf-8')
torch.save(dict(state_dict=model_b.state_dict(), config=CONFIG, vocabulary=vocab),
           out / 'notebook_weights.pt')
# This is an inference weights file, not the resumable checkpoint format of train.py.
saved = torch.load(out / 'notebook_weights.pt', weights_only=True, map_location='cpu')
restored = TinyTransformer(len(saved['vocabulary']), **saved['config'])
restored.load_state_dict(saved['state_dict']); restored.eval(); model_b.eval()
probe, _ = make_batch(train_data, CONFIG['context'], 1)
torch.testing.assert_close(restored(probe), model_b(probe))
print('Saved and reloaded matching weights.')
print('Import notebook-run/metrics.json into the dashboard: Train a small model → Inspect results.')
print('To resume full training with optimizer state, use the separate train.py workshop.')

## Explain what you built

1. Why does attention mix tokens while the feed-forward layer does not?
2. What do the two residual connections preserve?
3. Why is the target shifted by one character?
4. Which function computes gradients, and which changes weights?
5. Why can lower training loss coexist with worse held-out loss?
6. What happened when you changed learning rate? Use your actual numbers.
7. How is this small character model different from an instruction-tuned assistant?

Keep one failure, your diagnosis, your change, and the measured effect. That is useful interview evidence.